# MUS-Core на Kaggle — обучение 500M LM

**Многоцелевая Универсальная Система (МУС) / MUS-Agent**

Гибрид Rust + CUDA, FP16 training, gradient checkpointing.

### Рабочий процесс:
1. Установка Rust toolchain
2. Клонирование репозитория
3. Подготовка данных
4. Компиляция Rust+CUDA бинарника
5. Запуск обучения
6. Сохранение чекпоинтов

In [ ]:
import os, sys, subprocess, glob, shutil, json
from pathlib import Path

print("=== MUS-Core Kaggle Setup ===")
print(f"Python: {sys.version}")
print(f"CWD: {os.getcwd()}")
print(f"GPU: {subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout}")

### 1. Установка Rust

In [ ]:
%%bash
set -e
if ! command -v rustc &> /dev/null; then
    echo "Installing Rust..."
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    source "$HOME/.cargo/env"
    echo "Rust installed: $(rustc --version)"
else
    echo "Rust already installed: $(rustc --version)"
fi

# Verify CUDA
echo "CUDA: $(nvcc --version | tail -1)"

In [ ]:
# Source cargo env for this Python process
os.environ['PATH'] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ.get('PATH', '')
print(f"rustc: {subprocess.run(['rustc', '--version'], capture_output=True, text=True).stdout}")
print(f"cargo: {subprocess.run(['cargo', '--version'], capture_output=True, text=True).stdout}")

### 2. Клонирование репозитория

In [ ]:
%%bash
set -e
REPO="https://github.com/Slavawe/MUS-AI_platform.git"
TARGET_DIR="/kaggle/working/mus-core"

if [ ! -d "$TARGET_DIR" ]; then
    echo "Cloning $REPO..."
    git clone --depth 1 "$REPO" "$TARGET_DIR"
else
    echo "Repo already exists, pulling..."
    cd "$TARGET_DIR" && git pull
fi
echo "Cloned to $TARGET_DIR"
ls -la "$TARGET_DIR/mus-core-rust/"

### 3. Подготовка данных

Данные загружаются из Kaggle Dataset `mus-data` или генерируются через `prepare_dataset.py`.

In [ ]:
%%bash
set -e
cd /kaggle/working/mus-core

# Try Kaggle Dataset first
if [ -f /kaggle/input/mus-data/train_cache.bin ]; then
    echo "Using Kaggle Dataset: /kaggle/input/mus-data/"
    mkdir -p data
    cp /kaggle/input/mus-data/train_cache.bin data/
    cp /kaggle/input/mus-data/train_cache.meta data/ 2>/dev/null || true
    ls -lh data/
elif [ -f data/train_cache.bin ]; then
    echo "Using local data"
    ls -lh data/
else
    echo "Generating dataset via prepare_dataset.py..."
    pip install -q datasets tokenizers
    python prepare_dataset.py --max-samples 25000
    ls -lh data/
fi

### 4. Определение GPU архитектуры

Kaggle выдаёт P100 (sm_60) или T4 (sm_75). Устанавливаем правильный флаг.

In [ ]:
import subprocess
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                        capture_output=True, text=True).stdout.strip()
    print(f"GPU Compute Capability: {out}")
    major, minor = out.split('.')
    arch = f"sm_{major}{minor}"
except Exception as e:
    print(f"Detection failed: {e}, defaulting to sm_75")
    arch = "sm_75"

print(f"Target CUDA arch: {arch}")
os.environ['MUS_CUDA_ARCH'] = arch

### 5. Компиляция

In [ ]:
%%bash
set -e
cd /kaggle/working/mus-core/mus-core-rust
export MUS_CUDA_ARCH="$MUS_CUDA_ARCH"
echo "Building with arch=$MUS_CUDA_ARCH..."
cargo build --release 2>&1
echo "Build complete!"
ls -lh target/release/mus-core-rust

### 6. Запуск обучения

Чекпоинты сохраняются каждые 500 шагов и в конце каждой эпохи.
При перезапуске ноутбука обучение возобновится с последнего чекпоинта.

In [ ]:
%%bash
set -e
cd /kaggle/working/mus-core/mus-core-rust
export MUS_CKPT_DIR="/kaggle/working"

echo "Starting training..."
echo "Checkpoints: $MUS_CKPT_DIR/mus_checkpoint.bin"
echo ""

# Run training binary
timeout 32000 ./target/release/mus-core-rust ../data/train_cache.bin 2>&1 || \
    echo "Training interrupted (timeout or manual stop) - checkpoint saved"

# Verify checkpoint
if [ -f "$MUS_CKPT_DIR/mus_checkpoint.bin" ]; then
    CKPT_SIZE=$(ls -lh "$MUS_CKPT_DIR/mus_checkpoint.bin" | awk '{print $5}')
    echo "Checkpoint saved: $CKPT_SIZE"
fi

### 7. Результаты

Чекпоинты сохраняются в `/kaggle/working/mus_checkpoint.bin`. 
Для скачивания:
- Используйте Kaggle API: `kaggle kernels output <user>/<kernel-name>`
- Или скачайте вручную через UI (Data → Output)

Для возобновления обучения:
- Перезапустите ноутбук — он автоматически найдёт чекпоинт и продолжит с того же шага.

In [ ]:
%%bash
echo "=== Training Summary ==="
if [ -f /kaggle/working/mus_checkpoint.bin ]; then
    CKPT_SIZE=$(stat --format=%s /kaggle/working/mus_checkpoint.bin 2>/dev/null || \
                stat -f%z /kaggle/working/mus_checkpoint.bin 2>/dev/null)
    echo "Checkpoint: /kaggle/working/mus_checkpoint.bin ($(($CKPT_SIZE / 1024 / 1024)) MB)"
else
    echo "No checkpoint found"
fi
echo ""
echo "Files in /kaggle/working/:"
ls -lh /kaggle/working/ 2>/dev/null